# Create or Update Knowledge Assistant (Product Manuals)

Creates (or reuses) the `powertools-manuals-ka` **Agent Bricks Knowledge Assistant** and
points it at the dedicated `productmanuals` UC Volume for RAG Q&A over the 12
Bosch power-tool operating manuals. The KA does its own chunking / embedding / retrieval
over the PDFs — no `ai_parse_document`, no Vector Search index.

`display_name` is unique per workspace and is the idempotency key: if a KA with this name
already exists it is **reused** (its knowledge source is re-synced) — never re-created or
deleted. On the current FEVM build that is `knowledge-assistants/44e78d1c-…`.

Run this notebook **in the workspace** (it uses ambient `WorkspaceClient()` auth). Requires
`databricks-sdk`. Docs: [Knowledge Assistant SDK](https://databricks-sdk-py.readthedocs.io/).

In [ ]:
# %pip install databricks-sdk -q
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.knowledgeassistants import (
    FilesSpec,
    KnowledgeAssistant,
    KnowledgeSource,
)

w = WorkspaceClient()

# ── target (FEVM defaults) ─────────────────────────────────────────────────
CATALOG = "nikks_fevm_workspace_7405607030687545"
SCHEMA = "techsummit"
# Dedicated MANAGED Volume for the operating manuals (split out of the old
# raw_docs Volume). Manuals live at the Volume ROOT — no manuals/ subfolder.
# The sibling `datasheets` Volume (IDP source) is intentionally NOT given to
# the KA, so it stays grounded only on manuals.
VOLUME = "productmanuals"

# Guardrail: this demo operates ONLY in techsummit, never cdp.
assert SCHEMA == "techsummit", f"schema must be techsummit, got {SCHEMA!r}"
assert "cdp" not in (CATALOG.lower(), SCHEMA.lower()), "refusing to touch cdp"

DOCUMENTS_VOLUME_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/"

# ── KA config (real-manual text; display_name is the reuse key) ─────────────
DISPLAY_NAME = "powertools-manuals-ka"
DESCRIPTION = (
    "Answers questions about real Bosch power-tool operating manuals (safety, "
    "specifications, operation, battery/charging or mains, maintenance, "
    "troubleshooting, warranty) for the 12 demo power tools. RAG over the PDFs in "
    "the productmanuals Volume. A few tools are covered by their nearest-variant "
    "or family manual (e.g. psr-1080-li uses the Bosch PSB 1080 LI-2 booklet)."
)
INSTRUCTIONS = (
    "Answer only from the retrieved product manuals and always cite the source "
    "manual. Identify the specific tool model (e.g. GBH 2-26) the question is "
    "about. If a spec or fault code is not in the manuals, say so rather than "
    "guessing. A few tools are documented by their nearest-variant manual "
    "(e.g. psr-1080-li -> Bosch PSB 1080 LI-2); cite the actual manual retrieved."
)
SOURCE_DISPLAY_NAME = "powertools-pdf-manuals"
SOURCE_DESCRIPTION = (
    "Bosch power-tool operating manuals (PDFs) — safety, specs, operation, "
    "battery/mains, maintenance, troubleshooting, warranty."
)

print(f"DISPLAY_NAME={DISPLAY_NAME}")
print(f"DOCUMENTS_VOLUME_PATH={DOCUMENTS_VOLUME_PATH}")

In [ ]:
# Find an existing KA by display name (unique per workspace) — reuse if present.
ka_id = None
for a in w.knowledge_assistants.list_knowledge_assistants():
    if (a.display_name or "").strip() == DISPLAY_NAME:
        ka_id = (a.name or "").split("/", 1)[-1]
        break

if ka_id is None:
    ka = w.knowledge_assistants.create_knowledge_assistant(
        knowledge_assistant=KnowledgeAssistant(
            display_name=DISPLAY_NAME,
            description=DESCRIPTION,
            instructions=INSTRUCTIONS,
        )
    )
    ka_id = (ka.name or "").split("/", 1)[-1]
    print(f"Knowledge Assistant created: {ka_id}")
    src = w.knowledge_assistants.create_knowledge_source(
        parent=f"knowledge-assistants/{ka_id}",
        knowledge_source=KnowledgeSource(
            display_name=SOURCE_DISPLAY_NAME,
            description=SOURCE_DESCRIPTION,
            source_type="files",
            files=FilesSpec(path=DOCUMENTS_VOLUME_PATH),
        ),
    )
    print(f"Knowledge Source (files) created: {(src.name or '').rsplit('/', 1)[-1]} -> {DOCUMENTS_VOLUME_PATH}")
else:
    # Reuse the existing KA, but RECONCILE its knowledge-source path. A files
    # source's path is fixed at creation, so when the Volume moved (raw_docs/
    # manuals -> productmanuals) we repoint by creating a source at the new
    # path and then dropping any stale ones. Order matters: the API refuses to
    # delete the last remaining source ("must have at least one data source"),
    # so we CREATE-BEFORE-DELETE. Idempotent: re-runs find the source already
    # at DOCUMENTS_VOLUME_PATH and only prune leftovers.
    parent = f"knowledge-assistants/{ka_id}"
    print(f"Knowledge Assistant already exists (id={ka_id}). Reconciling source path -> {DOCUMENTS_VOLUME_PATH}")
    existing = list(w.knowledge_assistants.list_knowledge_sources(parent=parent))

    def _src_path(s):
        return getattr(getattr(s, "files", None), "path", None)

    target = next((s for s in existing if _src_path(s) == DOCUMENTS_VOLUME_PATH), None)
    if target is None:
        w.knowledge_assistants.create_knowledge_source(
            parent=parent,
            knowledge_source=KnowledgeSource(
                display_name=SOURCE_DISPLAY_NAME,
                description=SOURCE_DESCRIPTION,
                source_type="files",
                files=FilesSpec(path=DOCUMENTS_VOLUME_PATH),
            ),
        )
        print(f"  created source -> {DOCUMENTS_VOLUME_PATH}")
    else:
        print(f"  source already present at {DOCUMENTS_VOLUME_PATH}")

    # Prune any files source pointing elsewhere (e.g. the old raw_docs/manuals/).
    for s in existing:
        p = _src_path(s)
        if getattr(s, "files", None) is not None and p != DOCUMENTS_VOLUME_PATH:
            w.knowledge_assistants.delete_knowledge_source(name=s.name)
            print(f"  deleted stale source ({p})")

# Always sync so freshly-uploaded manuals get (re)indexed.
w.knowledge_assistants.sync_knowledge_sources(name=f"knowledge-assistants/{ka_id}")
print("Knowledge sources sync triggered.")

In [ ]:
# Status: a full re-index of the real, multi-hundred-page manuals takes ~10-15 min.
ka = w.knowledge_assistants.get_knowledge_assistant(name=f"knowledge-assistants/{ka_id}")
print("Knowledge Assistant state:", ka.state, "| endpoint:", ka.endpoint_name)
for s in w.knowledge_assistants.list_knowledge_sources(parent=f"knowledge-assistants/{ka_id}"):
    path = getattr(getattr(s, "files", None), "path", None)
    print("  source:", s.display_name, "| state:", s.state, "| path:", path)